# ShopVite FAQ Assistant — Démonstration

Ce notebook démontre le pipeline RAG en action sur plusieurs requêtes :
- 2 questions **dans le scope** (couvertes par la documentation)
- 1 question **hors-scope** (non couverte)
- 1 question **multi-thème** pour tester les limites

In [3]:
import requests
import json

BASE_URL = "http://localhost:8000"

def ask(question: str) -> dict:
    """Envoie une question à l'API et affiche la réponse formatée."""
    resp = requests.post(f"{BASE_URL}/ask", json={"question": question})
    data = resp.json()
    print(f"Question : {question}")
    print(f"Réponse  : {data['answer']}")
    print(f"Sources  : {data['sources']}")
    print(f"Confiance: {data['confidence']}")
    print("-" * 80)
    return data

## 1. Vérification du statut de l'API

In [4]:
health = requests.get(f"{BASE_URL}/health").json()
print(json.dumps(health, indent=2))

{
  "status": "ok",
  "vectorstore_loaded": true,
  "num_documents": 54
}


## 2. Question in-scope : Politique de retour

In [5]:
r1 = ask("Quelle est votre politique de retour ?")

Question : Quelle est votre politique de retour ?
Réponse  : La politique de retour de ShopVite permet de retourner tout produit acheté dans un délai de 30 jours calendaires à compter de la date de réception. Passé ce délai, aucun retour ne sera accepté, sauf en cas de défaut de fabrication couvert par la garantie. 

Pour être éligible à un retour, le produit doit être dans son emballage d'origine, non ouvert ou en état neuf, inclure tous les accessoires, manuels et câbles fournis, ne pas présenter de traces d'utilisation, de rayures ou de dommages causés par le client, et être accompagné du reçu ou de la preuve d'achat.

Le retour est gratuit pour les produits défectueux, tandis que pour les retours par convenance personnelle, des frais de retour de 9,95 $ seront déduits du remboursement. Une fois le produit reçu et inspecté, le remboursement est effectué selon le mode de paiement utilisé. [Source : politique_retour.txt]
Sources  : ['politique_retour.txt']
Confiance: high
------------

## 3. Question in-scope : Prix d'un produit

In [6]:
r2 = ask("Combien coûte le casque SoundMax Pro ANC ?")

Question : Combien coûte le casque SoundMax Pro ANC ?
Réponse  : Le casque SoundMax Pro ANC coûte 349,99 CAD. [Source : guide_produits.json]
Sources  : ['guide_produits.json']
Confiance: high
--------------------------------------------------------------------------------


## 4. Question in-scope : Livraison

In [7]:
r3 = ask("Quels sont les délais et coûts de livraison express au Canada ?")

Question : Quels sont les délais et coûts de livraison express au Canada ?
Réponse  : La livraison express au Canada coûte 14,95 $ et est effectuée en 2 à 3 jours ouvrables. [Source : livraison_expedition.md, sections 1]
Sources  : ['livraison_expedition.md']
Confiance: high
--------------------------------------------------------------------------------


## 5. Question hors-scope

In [8]:
r4 = ask("Quel est le prix du Bitcoin aujourd'hui ?")

Question : Quel est le prix du Bitcoin aujourd'hui ?
Réponse  : Je suis désolé, je ne dispose pas de cette information dans la documentation ShopVite. Je vous invite à contacter notre service client à support@shopvite.ca ou au 1-800-555-SHOP (7467) pour obtenir une réponse précise.
Sources  : ['conditions_generales.txt', 'guide_produits.json']
Confiance: low
--------------------------------------------------------------------------------


## 6. Résumé des résultats

| # | Question | Confiance | Sources |
|---|----------|-----------|----------|
| 1 | Politique de retour | high | politique_retour.txt |
| 2 | Prix SoundMax Pro ANC | high | guide_produits.json |
| 3 | Livraison express Canada | high | livraison_expedition.md |
| 4 | Prix du Bitcoin | low | (hors-scope) |